# BIDS Conversion of the fMRI Course DICOM Dataset (Neurodesk + HeuDiConv)

Convert the raw DICOM dataset at
`/data/Documents/fMRIcourse/fmri-course-data2026/mri/fmri-course-data2026_DICOM/`
into a [BIDS](https://bids.neuroimaging.io/)-compliant dataset using the
Neurodesk [`heudiconv`](https://neurodesk.org/edu/examples/workflows/bids_conversion.html)
container (which bundles `dcm2niix`).

**Input layout observed**
```
fmri-course-data2026_DICOM/
├── 7777CB-<SUBJECT>-<SESSION>-<DATE>-<TIME>-<SERIES#>-<DESC>-<NSLICES>_DICOM_images/
│       *.IMA        (Siemens DICOMs for ONE series)
├── ...              (one folder per series; all series for one subject share <SUBJECT>)
└── *_DICOM_images.zip
```

**Workflow**
1. Load the `heudiconv` module from Neurodesk.
2. Inventory subjects by grouping series folders on the `<SUBJECT>` field.
3. First pass: run HeuDiConv with the built-in `convertall` heuristic to get `dicominfo.tsv`.
4. Write a project-specific heuristic.
5. Second pass: convert every subject.
6. Validate.

## 1. Load the Neurodesk `heudiconv` module

Neurodesk exposes applications as Lmod modules backed by Singularity/Apptainer
containers. Loading `heudiconv/<ver>` puts both `heudiconv` and `dcm2niix`
on the `PATH` — no pip / conda needed.

See <https://neurodesk.org/edu/examples/workflows/bids_conversion.html>.

In [1]:
import module
await module.load('heudiconv')
await module.load('dcm2niix')
await module.list()

['heudiconv/1.3.1', 'dcm2niix/v1.0.20240202']

In [2]:
import shutil
print("heudiconv ->", shutil.which("heudiconv"))
print("dcm2niix  ->", shutil.which("dcm2niix"))

heudiconv -> /cvmfs/neurodesk.ardc.edu.au/containers/heudiconv_1.3.1_20241105/heudiconv
dcm2niix  -> /cvmfs/neurodesk.ardc.edu.au/containers/dcm2niix_v1.0.20240202_20241125/dcm2niix


In [3]:
import json
import re
import subprocess
import zipfile
from collections import defaultdict
from pathlib import Path

DICOM_DIR = Path(
    "/data/Documents/fMRIcourse/fmri-course-data2026/mri/"
    "fmri-course-data2026_DICOM"
)

PROJECT_DIR = Path.home() / "Desktop" / "fmri_course_JLU"
BIDS_DIR = PROJECT_DIR / "bids"
HEURISTIC = PROJECT_DIR / "heuristic.py"

BIDS_DIR.mkdir(parents=True, exist_ok=True)
assert DICOM_DIR.exists(), f"DICOM directory not found: {DICOM_DIR}"

## 2. Inventory subjects

Each `*_DICOM_images` folder is **one series** (not one subject). All series
for a given subject share the second dash-segment of the folder name:

```
7777CB-2604201-01CB-20260420-1335-01-AAHead_Scout-128_DICOM_images
       ^^^^^^^ subject
```


In [4]:
def unzip_if_needed(zip_path: Path) -> Path:
    target = zip_path.with_suffix("")
    if target.exists():
        return target
    print(f"Unzipping {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(zip_path.parent)
    return target


for zp in sorted(DICOM_DIR.glob("*_DICOM_images.zip")):
    unzip_if_needed(zp)

series_dirs = sorted(p for p in DICOM_DIR.glob("*_DICOM_images") if p.is_dir())


def subject_id_from_folder(name: str) -> str:
    parts = name.split("-")
    return re.sub(r"[^A-Za-z0-9]", "", parts[1]) if len(parts) > 1 else ""


SUBJECTS: dict[str, list[Path]] = defaultdict(list)
for p in series_dirs:
    sid = subject_id_from_folder(p.name)
    if sid:
        SUBJECTS[sid].append(p)
SUBJECTS = dict(SUBJECTS)

for label, paths in SUBJECTS.items():
    print(f"sub-{label}  ({len(paths)} series)")

sub-2604201  (14 series)


## 3. First pass — `convertall` on one subject

We use the `-d` template with `{subject}` so HeuDiConv picks up every
series folder for a subject and every `.IMA` inside, without blowing up
argv.

In [5]:
DICOM_TEMPLATE = str(DICOM_DIR / "7777CB-{subject}-*_DICOM_images" / "*.IMA")
print("DICOM template:", DICOM_TEMPLATE)

first_label = next(iter(SUBJECTS))

cmd = [
    "heudiconv",
    "-d", DICOM_TEMPLATE,
    "-o", str(BIDS_DIR),
    "-f", "convertall",
    "-s", first_label,
    "-c", "none",
    "--overwrite",
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-1500:])
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-2000:])
    raise RuntimeError("heudiconv convertall failed")

DICOM template: /data/Documents/fMRIcourse/fmri-course-data2026/mri/fmri-course-data2026_DICOM/7777CB-{subject}-*_DICOM_images/*.IMA



In [6]:
import pandas as pd

info_dir = BIDS_DIR / ".heudiconv" / first_label / "info"
dicominfo_tsv = next(info_dir.glob("dicominfo*.tsv"), None)
print("Info file:", dicominfo_tsv)

df = pd.read_csv(dicominfo_tsv, sep="\t")
cols = [
    "series_id", "series_description", "protocol_name",
    "dim1", "dim2", "dim3", "dim4", "TR", "TE", "image_type",
]
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(df[cols])

Info file: /home/jovyan/Desktop/fmri_course_JLU/bids/.heudiconv/2604201/info/dicominfo.tsv


,series_id,series_description,protocol_name,dim1,dim2,dim3,dim4,TR,TE,image_type
0,1-AAHead_Scout,AAHead_Scout,AAHead_Scout,160,160,128,1,0.00315,1.37,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM')"
1,2-AAHead_Scout,AAHead_Scout_MPR_sag,AAHead_Scout,162,162,5,1,0.00315,1.37,"('DERIVED', 'PRIMARY', 'MPR', 'ND', 'NORM')"
2,3-AAHead_Scout,AAHead_Scout_MPR_cor,AAHead_Scout,162,162,3,1,0.00315,1.37,"('DERIVED', 'PRIMARY', 'MPR', 'ND', 'NORM')"
3,4-AAHead_Scout,AAHead_Scout_MPR_tra,AAHead_Scout,162,162,3,1,0.00315,1.37,"('DERIVED', 'PRIMARY', 'MPR', 'ND', 'NORM')"
4,5-t1_mprage_sag_p3_iso,t1_mprage_sag_p3_iso,t1_mprage_sag_p3_iso,256,256,176,1,1.58000,2.30,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM')"
5,6-gre_field_mapping_DistFact,gre_field_mapping_DistFact,gre_field_mapping_DistFact,80,80,90,1,0.53100,7.38,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM')"
6,7-gre_field_mapping_DistFact,gre_field_mapping_DistFact,gre_field_mapping_DistFact,80,80,45,1,0.53100,7.38,"('ORIGINAL', 'PRIMARY', 'P', 'ND')"
7,8-ep2d_bold_1_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_1_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_1_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,80,80,63,200,1.50000,29.00,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM', 'MOSAIC')"
8,9-ep2d_bold_2_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_2_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_2_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,80,80,63,160,1.50000,29.00,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM', 'MOSAIC')"
9,10-ep2d_bold_3_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_3_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,ep2d_bold_3_p13_2.4_2.4_2.4_mm_50sl_TR1500_1,80,80,63,184,1.50000,29.00,"('ORIGINAL', 'PRIMARY', 'M', 'ND', 'NORM', 'MOSAIC')"


## 4. Write the project heuristic

| Protocol | BIDS mapping |
|---|---|
| `AAHead_Scout*` | **skipped** (localizer) |
| `t1_mprage_sag_p3_iso` | `anat/sub-<id>_T1w` |
| `ep2d_bold_<N>_p13_...` | `func/sub-<id>_task-main_run-<N>_bold` |
| `gre_field_mapping_DistFact-*` | `fmap/sub-<id>_run-<k>_fieldmap` |
| `PhoenixZIPReport` | **skipped** |

In [7]:
HEURISTIC.write_text('''\
"""HeuDiConv heuristic for the fMRI Course JLU 2026 dataset."""

import re


def create_key(template, outtype=("nii.gz",), annotation_classes=None):
    if not template:
        raise ValueError("Template must be a valid format string")
    return template, outtype, annotation_classes


def infotodict(seqinfo):
    t1w = create_key("sub-{subject}/anat/sub-{subject}_T1w")
    bold = create_key(
        "sub-{subject}/func/sub-{subject}_task-main_run-{item:02d}_bold"
    )
    fmap_mag = create_key("sub-{subject}/fmap/sub-{subject}_magnitude")
    fmap_phase = create_key("sub-{subject}/fmap/sub-{subject}_phasediff")

    info = {t1w: [], bold: [], fmap_mag: [], fmap_phase: []}
    bold_series = []

    for s in seqinfo:
        proto = (s.protocol_name or "").lower()
        image_type = tuple(str(x).upper() for x in (s.image_type or ()))

        # Skip localizers and scanner reports.
        if "aahead_scout" in proto or "phoenix" in proto:
            continue

        # Anatomical T1w.
        if "mprage" in proto:
            info[t1w].append(s.series_id)
            continue

        # Task BOLD (ep2d_bold_<N>_...). Sort by the numeric index embedded
        # in the protocol name so run-01..run-06 follow acquisition order.
        m = re.match(r"ep2d_bold_(\\d+)", proto)
        if m:
            bold_series.append((int(m.group(1)), s.series_id))
            continue

        # Siemens gre_field_mapping: magnitude (image_type contains "M")
        # vs phase-difference (image_type contains "P").
        if "gre_field_mapping" in proto or "field_map" in proto:
            if "P" in image_type:
                info[fmap_phase].append(s.series_id)
            else:
                info[fmap_mag].append(s.series_id)
            continue

    for _, sid in sorted(bold_series):
        info[bold].append(sid)

    return info
''')
print(f"Wrote heuristic to {HEURISTIC}")

Wrote heuristic to /home/jovyan/Desktop/fmri_course_JLU/heuristic.py


## 5. Second pass — convert every subject

In [8]:
def reset_subject_state(label: str) -> None:
    """Clear cached heuristic output and any flat convertall leftovers.

    HeuDiConv caches the mapping produced by ``infotodict`` in
    ``.heudiconv/<label>/info/{label}.auto.txt``. If that file already exists
    from a previous ``convertall`` run, a subsequent run with a new
    heuristic will reuse the cached mapping instead of re-executing
    ``infotodict`` — and output will land as flat ``runNNN.nii.gz`` files
    in the BIDS root. Deleting those two text files forces a fresh run.
    """
    info_dir = BIDS_DIR / ".heudiconv" / label / "info"
    for pattern in (f"{label}.auto.txt", f"{label}.edit.txt"):
        for p in info_dir.glob(pattern):
            p.unlink()
    for pattern in ("run*.nii.gz", "run*.json"):
        for p in BIDS_DIR.glob(pattern):
            p.unlink()


def convert_subject(label: str) -> None:
    reset_subject_state(label)
    cmd = [
        "heudiconv",
        "-d", DICOM_TEMPLATE,
        "-o", str(BIDS_DIR),
        "-f", str(HEURISTIC),
        "-s", label,
        "-c", "dcm2niix",
        "-b", "--minmeta",
        "--overwrite",
    ]
    print(">>>", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-2000:])
        raise RuntimeError(f"heudiconv failed for sub-{label}")


for label in SUBJECTS:
    convert_subject(label)

subprocess.run(["chmod", "-R", "u+w", str(BIDS_DIR)], check=True)

>>> heudiconv -d /data/Documents/fMRIcourse/fmri-course-data2026/mri/fmri-course-data2026_DICOM/7777CB-{subject}-*_DICOM_images/*.IMA -o /home/jovyan/Desktop/fmri_course_JLU/bids -f /home/jovyan/Desktop/fmri_course_JLU/heuristic.py -s 2604201 -c dcm2niix -b --minmeta --overwrite
6-04-30T12:23:38.614260:Compress: "/usr/bin/pigz" -n -f -6 "/home/jovyan/Desktop/fmri_course_JLU/bids/sub-2604201/fmap/sub-2604201_magnitude_heudiconv563_e1.nii"
260430-12:23:38,614 nipype.interface INFO:
	 stdout 2026-04-30T12:23:38.614260:Convert 45 DICOM as /home/jovyan/Desktop/fmri_course_JLU/bids/sub-2604201/fmap/sub-2604201_magnitude_heudiconv563_e2 (80x80x45x1)
260430-12:23:38,660 nipype.interface INFO:
	 stdout 2026-04-30T12:23:38.660220:Compress: "/usr/bin/pigz" -n -f -6 "/home/jovyan/Desktop/fmri_course_JLU/bids/sub-2604201/fmap/sub-2604201_magnitude_heudiconv563_e2.nii"
260430-12:23:38,660 nipype.interface INFO:
	 stdout 2026-04-30T12:23:38.660220:Conversion required 0.184869 seconds (0.049452 for co

CompletedProcess(args=['chmod', '-R', 'u+w', '/home/jovyan/Desktop/fmri_course_JLU/bids'], returncode=0)

In [9]:
(BIDS_DIR / "dataset_description.json").write_text(json.dumps({
    "Name": "fMRI Course JLU 2026",
    "BIDSVersion": "1.9.0",
    "DatasetType": "raw",
    "Authors": ["fMRI Course JLU"],
}, indent=2))
print((BIDS_DIR / "dataset_description.json").read_text())

{
  "Name": "fMRI Course JLU 2026",
  "BIDSVersion": "1.9.0",
  "DatasetType": "raw",
  "Authors": [
    "fMRI Course JLU"
  ]
}


## 6. Validate

In [10]:
if shutil.which("bids-validator"):
    cmd = ["bids-validator", str(BIDS_DIR)]
elif shutil.which("npx"):
    cmd = ["npx", "--yes", "bids-validator", str(BIDS_DIR)]
else:
    cmd = None
    print("No BIDS validator found.\n"
          "Tip: load the Neurodesk module with `%module load bidsvalidator`,\n"
          "or use https://bids-standard.github.io/bids-validator/")

if cmd is not None:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-2000:])

run-02_events.json
		./sub-2604201/func/sub-2604201_task-main_run-03_events.tsv
			Evidence: Columns: TODO -- fill in rows and add more tab-separated columns if desired not defined, please define in: /events.json, /task-main_events.json,/run-03_events.json,/task-main_run-03_events.json,/sub-2604201/sub-2604201_events.json,/sub-2604201/sub-2604201_task-main_events.json,/sub-2604201/sub-2604201_run-03_events.json,/sub-2604201/sub-2604201_task-main_run-03_events.json,/sub-2604201/func/sub-2604201_events.json,/sub-2604201/func/sub-2604201_task-main_events.json,/sub-2604201/func/sub-2604201_run-03_events.json,/sub-2604201/func/sub-2604201_task-main_run-03_events.json
		./sub-2604201/func/sub-2604201_task-main_run-04_events.tsv
			Evidence: Columns: TODO -- fill in rows and add more tab-separated columns if desired not defined, please define in: /events.json, /task-main_events.json,/run-04_events.json,/task-main_run-04_events.json,/sub-2604201/sub-2604201_events.json,/sub-2604201/sub-2604201

In [11]:
from bids import BIDSLayout

layout = BIDSLayout(BIDS_DIR, validate=False)
print(layout)
print("\nSubjects:", layout.get_subjects())
print("Tasks:   ", layout.get_tasks())
print("\nFiles:")
for f in layout.get(extension=["nii", "nii.gz"]):
    print("  ", f.relpath)

BIDS Layout: ...n/Desktop/fmri_course_JLU/bids | Subjects: 1 | Sessions: 0 | Runs: 6

Subjects: ['2604201']
Tasks:    ['main']

Files:
   sub-2604201/anat/sub-2604201_T1w.nii.gz
   sub-2604201/fmap/sub-2604201_magnitude1.nii.gz
   sub-2604201/fmap/sub-2604201_magnitude2.nii.gz
   sub-2604201/fmap/sub-2604201_phasediff.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-01_bold.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-02_bold.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-03_bold.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-04_bold.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-05_bold.nii.gz
   sub-2604201/func/sub-2604201_task-main_run-06_bold.nii.gz
